# Package

In [20]:
# ----------------------------
# Core
# ----------------------------
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# Feature Store
# ----------------------------
from feast import FeatureStore

# ----------------------------
# Nixtla MLForecast
# ----------------------------
from mlforecast import MLForecast
from mlforecast.utils import PredictionIntervals

# ----------------------------
# Model backend
# ----------------------------
from sklearn.linear_model import LinearRegression

# Importation des données

# Utilisation des données lags 12

In [21]:
# ----------------------------
# Locate Feast repo (notebook-safe)
# ----------------------------
from pathlib import Path
import pandas as pd
from feast import FeatureStore

def find_project_root(start: Path, marker: str = "2_data_processing") -> Path:
    p = start.resolve()
    for parent in [p] + list(p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"Impossible de trouver la racine projet (marker '{marker}') depuis {start}"
    )

PROJECT_ROOT = find_project_root(Path.cwd(), marker="2_data_processing")

FEAST_REPO_PATH = (
    PROJECT_ROOT
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

print("FEAST_REPO_PATH:", FEAST_REPO_PATH)
print("feature_store.yaml exists:", (FEAST_REPO_PATH / "feature_store.yaml").exists())

# ----------------------------
# Load features from Feast
# ----------------------------
def load_features_from_feast(entity_df: pd.DataFrame, feature_refs: list[str]) -> pd.DataFrame:
    fs = FeatureStore(repo_path=str(FEAST_REPO_PATH))
    return fs.get_historical_features(
        entity_df=entity_df,
        features=feature_refs,
        full_feature_names=True,
    ).to_df()

# ----------------------------
# Config data
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"

series_ids = [
    "BUSLOANS",
    "CPIAUCSL",
    "DPCERA3M086SBEA",
    "INDPRO",
    "M2SL",
    "OILPRICEX",
    "RPI",
    "SP500",
    "TB3MS",
    "UNRATE",
    "USREC",
]

# On veut les séries stationnarisées pour le modèle
FEATURE_REFS = ["stationary_value:value"]

# ----------------------------
# 1) Dates de référence (via UNRATE raw) pour avoir le vrai calendrier dispo
# ----------------------------
calendar = pd.date_range(start=START, end=END, freq=FREQ)

entity_df_unrate = pd.DataFrame({"series_id": ["UNRATE"] * len(calendar), "date": calendar})
df_unrate_dates = load_features_from_feast(entity_df_unrate, ["raw_value:value"])

dates = (
    pd.to_datetime(df_unrate_dates["date"], utc=True, errors="coerce")
      .dt.tz_convert(None)
      .dropna()
      .sort_values()
      .unique()
)

print("Nb dates:", len(dates))
print("Date min:", dates.min(), "| Date max:", dates.max())

# ----------------------------
# 2) Entity DF multi-séries × dates (long)
# ----------------------------
entity_df = (
    pd.MultiIndex.from_product([series_ids, dates], names=["series_id", "date"])
      .to_frame(index=False)
)

# ----------------------------
# 3) Fetch stationary features (long)
# ----------------------------
df_stationary = load_features_from_feast(entity_df, FEATURE_REFS)

df_stationary["date"] = (
    pd.to_datetime(df_stationary["date"], utc=True, errors="coerce")
      .dt.tz_convert(None)
)

value_col = "stationary_value__value"
if value_col not in df_stationary.columns:
    candidates = [c for c in df_stationary.columns if c.endswith("__value")]
    raise KeyError(f"Expected '{value_col}' not found. Candidates: {candidates}")

df_stationary = (
    df_stationary[["series_id", "date", value_col]]
    .rename(columns={value_col: "value"})
)

print("df_stationary shape:", df_stationary.shape)
print(df_stationary.head())

# ----------------------------
# 4) Dataset régression ciblé UNRATE
#    y = UNRATE (stationary)
#    exog = autres séries (contemporaines)
# ----------------------------
df_y = (
    df_stationary[df_stationary["series_id"] == "UNRATE"]
    .sort_values("date")
    .rename(columns={"value": "y"})
    .reset_index(drop=True)
)

df_x_long = df_stationary[df_stationary["series_id"] != "UNRATE"].copy()
df_x_long = df_x_long[df_x_long["date"].isin(df_y["date"])]

df_x = (
    df_x_long
    .pivot_table(index="date", columns="series_id", values="value", aggfunc="last")
    .reset_index()
)

df_model = (
    df_y[["date", "y"]]
    .merge(df_x, on="date", how="left")
    .dropna()
)

# ----------------------------
# 5) Format MLForecast (long + exog cols)
# ----------------------------
ts_lr = df_model.rename(columns={"date": "ds"})
ts_lr["unique_id"] = "UNRATE"

exog_cols = [c for c in ts_lr.columns if c not in ["unique_id", "ds", "y"]]
ts_lr = ts_lr[["unique_id", "ds", "y"] + exog_cols]

print("ts_lr shape:", ts_lr.shape)
print("Exog cols:", exog_cols)
ts_lr.head()

FEAST_REPO_PATH: D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\2_data_processing\feature_store\feast_repo\feature_repo
feature_store.yaml exists: True
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
Nb dates: 801
Date min: 1959-01-01 00:00:00 | Date max: 2025-09-01 00:00:00
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.
df_stationary shape: (8679, 3)
  series_id       date     value
0  BUSLOANS 1960-01-01  0.011578
1    INDPRO 1960-01-01  0.091976
2     USREC 1960-01-01  0.000000
3      M2SL 1960-01-01  0.001323
4  CPIAUCSL 1960-01-01 -0.006156
ts_lr shape: (788, 13)
Exog cols: ['BUSLOANS', 'CPIAUCSL', 'DPCERA3M086SBEA', 'INDPRO', 'M2SL', 'OILPRICEX', 'RPI', 'SP500', 'TB3MS', 'USREC']


,unique_id,ds,y,BUSLOANS,CPIAUCSL,DPCERA3M086SBEA,INDPRO,M2SL,OILPRICEX,RPI,SP500,TB3MS,USREC
0,UNRATE,1960-01-01,-0.8,0.011578,-0.006156,0.001204,0.091976,0.001323,0.0,0.020977,0.017909,0.30,0.0
1,UNRATE,1960-02-01,-1.1,0.011905,-0.003767,0.006009,0.076960,0.002007,0.0,0.014565,-0.025663,-0.19,0.0
2,UNRATE,1960-03-01,-0.2,-0.008356,-0.005455,0.021240,0.007959,0.001324,0.0,0.006250,-0.070857,-1.18,0.0
3,UNRATE,1960-04-01,0.0,-0.009098,0.005090,0.033752,-0.025916,0.000634,0.0,0.006489,-0.040442,-1.12,0.0
4,UNRATE,1960-05-01,0.0,-0.000359,0.003383,0.009040,-0.018119,0.003977,0.0,0.007747,-0.010090,-0.67,1.0


In [60]:
# ============================================================
# (EXTRACT) ts_ar depuis ts_lr (univarié pour AR12)
# ============================================================

import pandas as pd

# 1) Copie minimale: unique_id, ds, y uniquement
ts_ar = ts_lr[["unique_id", "ds", "y"]].copy()

# 2) Normaliser ds en début de mois (MS)
ts_ar["ds"] = (
    pd.to_datetime(ts_ar["ds"], errors="coerce")
      .dt.to_period("M")
      .dt.to_timestamp(how="start")
      .dt.normalize()
)

# 3) Checks
if ts_ar["ds"].isna().any():
    bad = ts_ar[ts_ar["ds"].isna()].head()
    raise ValueError(f"Dates 'ds' invalides après parsing. Exemples:\n{bad}")

# 4) Tri + (optionnel) unicité de la série
ts_ar = ts_ar.sort_values(["unique_id", "ds"]).reset_index(drop=True)

# (optionnel mais conseillé) vérifier 1 ligne par mois
dup = ts_ar.duplicated(subset=["unique_id", "ds"]).sum()
print("duplicates (unique_id, ds) in ts_ar:", dup)

print("ts_ar columns:", ts_ar.columns.tolist())
print("ts_ar ds range:", ts_ar["ds"].min().date(), "→", ts_ar["ds"].max().date())
ts_ar.head()


duplicates (unique_id, ds) in ts_ar: 0
ts_ar columns: ['unique_id', 'ds', 'y']
ts_ar ds range: 1960-01-01 → 2025-08-01


,unique_id,ds,y
0,UNRATE,1960-01-01,-0.8
1,UNRATE,1960-02-01,-1.1
2,UNRATE,1960-03-01,-0.2
3,UNRATE,1960-04-01,0.0
4,UNRATE,1960-05-01,0.0


# Dictionnaire de modèle

In [66]:
from mlforecast import MLForecast
from sklearn.linear_model import LinearRegression

MLF_MODELS = {
    "AR12": lambda freq: MLForecast(
        models={"AR12": LinearRegression()},
        freq=freq,
        lags=list(range(1, 13)),  # ✅ AR(12) = lags 1..12
        date_features=[],
    )
}

# Backtesting

In [67]:
import pandas as pd
from dateutil.relativedelta import relativedelta
from mlforecast.utils import PredictionIntervals

def _ensure_ms(x):
    """Force un Timestamp au 1er du mois (MS)."""
    x = pd.Timestamp(x)
    return x.to_period("M").to_timestamp(how="start").normalize()

def _n_windows_monthly(ds_start, ds_end):
    """Nombre de mois inclusifs entre ds_start et ds_end."""
    return (ds_end.year - ds_start.year) * 12 + (ds_end.month - ds_start.month) + 1

def run_backtesting_h12_monthly(
    mlf,
    ts,
    *,
    h=12,
    exp_start="1990-01-01",
    exp_end="2025-08-01",
    step_size=1,
    pi_windows=24,
    levels=[95],
    keep_last_cutoff=True,   # ✅ ds unique si True
):
    ts = ts.copy()

    # ds en MS (robuste)
    ts["ds"] = (
        pd.to_datetime(ts["ds"], errors="coerce")
          .dt.to_period("M")
          .dt.to_timestamp(how="start")
          .dt.normalize()
    )
    if ts["ds"].isna().any():
        bad = ts[ts["ds"].isna()].head()
        raise ValueError(f"Dates 'ds' invalides après parsing. Exemples:\n{bad}")

    # 1) bornes exactes de l’expérience
    exp_start = _ensure_ms(exp_start)
    exp_end   = _ensure_ms(exp_end)

    # 2) cutoffs correspondants
    cutoff_start = exp_start - relativedelta(months=h)
    cutoff_end   = exp_end   - relativedelta(months=h)

    # 3) partitions (cutoffs mensuels inclusifs)
    partitions = _n_windows_monthly(cutoff_start, cutoff_end)

    # 4) anti-fuite (recommandé)
    ts = ts.loc[ts["ds"] <= exp_end].copy()

    # Prediction intervals (conformal)
    pi = PredictionIntervals(h=h, n_windows=pi_windows, method="conformal_distribution")

    bkt_df = mlf.cross_validation(
        df=ts,
        h=h,
        step_size=step_size,
        n_windows=partitions,
        prediction_intervals=pi,
        level=levels,
        fitted=True,
        static_features=[],
    )

    # Filtrer la plage exacte ds
    bkt_df = bkt_df[(bkt_df["ds"] >= exp_start) & (bkt_df["ds"] <= exp_end)].copy()
    bkt_df = bkt_df.sort_values(["unique_id", "ds", "cutoff"]).reset_index(drop=True)

    # ✅ Option: 1 seule ligne par ds (dernier cutoff)
    if keep_last_cutoff:
        bkt_df = (
            bkt_df.sort_values(["unique_id", "ds", "cutoff"])
                  .groupby(["unique_id", "ds"], as_index=False)
                  .tail(1)
                  .reset_index(drop=True)
        )

    meta = {
        "h": h,
        "step_size": step_size,
        "exp_start": exp_start,
        "exp_end": exp_end,
        "cutoff_start": cutoff_start,
        "cutoff_end": cutoff_end,
        "partitions": partitions,
        "pi_windows": pi_windows,
        "keep_last_cutoff": keep_last_cutoff,
    }

    return bkt_df, meta

# Run 

In [70]:
# ============================================================
# RUN – AR(12) (Nixtla MLForecast) | UNIVARIATE (AR12 = OLS on lags 1..12)
# Mensuel (step=1) + horizon 12 + bornes exactes exp
# + sortie finale avec ds unique (dernier cutoff)
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from dateutil.relativedelta import relativedelta

PROJECT_ROOT = Path.cwd().parent
print("PROJECT_ROOT =", PROJECT_ROOT.resolve())

# -----------------------------
# Paramètres
# -----------------------------
H = 12
STEP_SIZE = 1
PI_WINDOWS = 3
LEVELS = [95]
LAGS = 12

EXP_START = pd.Timestamp("1990-01-01")
EXP_END   = pd.Timestamp("2025-08-01")   # <- jusqu'à août 2025 inclus

def _ensure_ms(x):
    x = pd.Timestamp(x)
    return x.to_period("M").to_timestamp(how="start").normalize()

def _n_windows_monthly(ds_start, ds_end):
    return (ds_end.year - ds_start.year) * 12 + (ds_end.month - ds_start.month) + 1

# -----------------------------
# Extraction ts_ar depuis ts_lr
# -----------------------------
ts_ar = ts_lr[["unique_id", "ds", "y"]].copy()

ts_ar["ds"] = (
    pd.to_datetime(ts_ar["ds"], errors="coerce")
      .dt.to_period("M")
      .dt.to_timestamp(how="start")
      .dt.normalize()
)

# y -> numeric + dropna (critique avec lags)
ts_ar["y"] = pd.to_numeric(ts_ar["y"], errors="coerce")

if ts_ar["ds"].isna().any():
    bad = ts_ar[ts_ar["ds"].isna()].head()
    raise ValueError(f"Certaines dates 'ds' n'ont pas pu être parsées. Exemples:\n{bad}")

ts_ar = ts_ar.dropna(subset=["y"]).copy()

# (optionnel) dédoublonnage mensuel
ts_ar = (
    ts_ar.sort_values(["unique_id", "ds"])
         .drop_duplicates(subset=["unique_id", "ds"], keep="last")
         .reset_index(drop=True)
)

# --- bornes exactes de l'expérience ---
EXP_START = _ensure_ms(EXP_START)
EXP_END   = _ensure_ms(EXP_END)

CUTOFF_START = EXP_START - relativedelta(months=H)  # 1989-01-01
CUTOFF_END   = EXP_END   - relativedelta(months=H)  # 2024-08-01
PARTITIONS   = _n_windows_monthly(CUTOFF_START, CUTOFF_END)

print("✅ EXP ds range          :", EXP_START.date(), "→", EXP_END.date())
print("✅ CUTOFF range          :", CUTOFF_START.date(), "→", CUTOFF_END.date())
print("✅ PARTITIONS (n_windows):", PARTITIONS)
print("ts_ar ds range           :", ts_ar["ds"].min().date(), "→", ts_ar["ds"].max().date())
print("ts_ar y NA count         :", int(ts_ar["y"].isna().sum()))

# optionnel (recommandé) : éviter fuite future
ts_ar = ts_ar[ts_ar["ds"] <= EXP_END].copy()

# -----------------------------
# Instancier le modèle AR12
# -----------------------------
mlf_ar12 = MLF_MODELS["AR12"](FREQ)

model_names = list(mlf_ar12.models.keys())
first_name = next(iter(mlf_ar12.models))
print("Running models:", model_names)
print("First model class:", mlf_ar12.models[first_name].__class__.__name__)
print("Freq:", FREQ)

# -----------------------------
# Backtesting
# -----------------------------
# ✅ Important: assez long pour (i) lags=12, (ii) conformal: h*pi_windows+1
MIN_TRAIN_N = max(60, LAGS + 1, H * PI_WINDOWS + 1)  # ex: max(60,13,37)=60
print("MIN_TRAIN_N:", MIN_TRAIN_N)

bkt_ar12 = run_backtesting_h12_simple(
    mlf=mlf_ar12,
    ts=ts_ar,
    h=H,
    step_size=STEP_SIZE,
    partitions=PARTITIONS,
    pi_windows=PI_WINDOWS,
    levels=LEVELS,
    min_train_n=MIN_TRAIN_N,
)

# -----------------------------
# Filtrer exactement l'expérience
# -----------------------------
bkt_ar12_eval = bkt_ar12[(bkt_ar12["ds"] >= EXP_START) & (bkt_ar12["ds"] <= EXP_END)].copy()
bkt_ar12_eval = bkt_ar12_eval.sort_values(["unique_id", "ds", "cutoff"]).reset_index(drop=True)

# ============================================================
# ✅ 1 seule ligne par ds : garder le cutoff le plus récent
# ============================================================
bkt_ar12_final = (
    bkt_ar12_eval
    .sort_values(["unique_id", "ds", "cutoff"])
    .groupby(["unique_id", "ds"], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

# -----------------------------
# Checks
# -----------------------------
print("bkt_ar12_eval rows       :", len(bkt_ar12_eval))
print("bkt_ar12_final rows      :", len(bkt_ar12_final))
print("bkt_ar12_final ds range  :", bkt_ar12_final["ds"].min().date(), "→", bkt_ar12_final["ds"].max().date())
print("duplicates (unique_id, ds):", bkt_ar12_final.duplicated(subset=["unique_id", "ds"]).sum())

bkt_ar12_final.head()

PROJECT_ROOT = D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\3_notebook
✅ EXP ds range          : 1990-01-01 → 2025-08-01
✅ CUTOFF range          : 1989-01-01 → 2024-08-01
✅ PARTITIONS (n_windows): 428
ts_ar ds range           : 1960-01-01 → 2025-08-01
ts_ar y NA count         : 0
Running models: ['AR12']
First model class: LinearRegression
Freq: MS
MIN_TRAIN_N: 60
bkt_ar12_eval rows       : 5070
bkt_ar12_final rows      : 428
bkt_ar12_final ds range  : 1990-01-01 → 2025-08-01
duplicates (unique_id, ds): 0


,unique_id,ds,cutoff,y,AR12,AR12-lo-95,AR12-hi-95
0,UNRATE,1990-01-01,1989-12-01,0.0,-0.104865,-1.768264,1.558534
1,UNRATE,1990-02-01,1990-01-01,0.1,0.021839,-1.826977,1.870656
2,UNRATE,1990-03-01,1990-02-01,0.2,0.211108,-3.853666,4.275882
3,UNRATE,1990-04-01,1990-03-01,0.2,0.052910,-1.881681,1.987502
4,UNRATE,1990-05-01,1990-04-01,0.2,0.242059,-0.127650,0.611769


# Sauvegarde

In [71]:
# ============================================================
# (ADD) Build standardized OOS forecast table (AR12) — SAFE
# ============================================================

model_col = "AR12"
src = bkt_ar12_final  # ✅ ds unique (recommandé)

# --- find PI column names robustly ---
candidate_lo = [f"{model_col}-lo-95", f"{model_col}-lo-95.0"]
candidate_hi = [f"{model_col}-hi-95", f"{model_col}-hi-95.0"]

lo_col = next((c for c in candidate_lo if c in src.columns), None)
hi_col = next((c for c in candidate_hi if c in src.columns), None)

if lo_col is None or hi_col is None:
    raise KeyError(
        f"Prediction interval columns not found for {model_col}. "
        f"Available columns: {list(src.columns)}"
    )

df_ar12_forecasts = (
    src[["unique_id", "ds", "cutoff", "y", model_col, lo_col, hi_col]]
    .rename(columns={
        "unique_id": "series_id",
        "ds": "date",
        "y": "y_obs",
        model_col: "y_hat_ar12",
        lo_col: "y_hat_ar12_lo_95",
        hi_col: "y_hat_ar12_hi_95",
    })
    .sort_values(["series_id", "date"])
    .reset_index(drop=True)
)

print("df_ar12_forecasts shape:", df_ar12_forecasts.shape)
print("duplicates (series_id, date):", df_ar12_forecasts.duplicated(["series_id","date"]).sum())
df_ar12_forecasts.head()

df_ar12_forecasts shape: (428, 7)
duplicates (series_id, date): 0


,series_id,date,cutoff,y_obs,y_hat_ar12,y_hat_ar12_lo_95,y_hat_ar12_hi_95
0,UNRATE,1990-01-01,1989-12-01,0.0,-0.104865,-1.768264,1.558534
1,UNRATE,1990-02-01,1990-01-01,0.1,0.021839,-1.826977,1.870656
2,UNRATE,1990-03-01,1990-02-01,0.2,0.211108,-3.853666,4.275882
3,UNRATE,1990-04-01,1990-03-01,0.2,0.052910,-1.881681,1.987502
4,UNRATE,1990-05-01,1990-04-01,0.2,0.242059,-0.127650,0.611769


# Graphique

In [ ]:
import pandas as pd
from utilsforecast.plotting import plot_series

# ============================================================
# SAFE prepare df_obs / df_fcst for utilsforecast.plot_series
# ============================================================

# 0) assure datetime + tri
df_plot = df_ar12_forecasts.copy()
df_plot["date"] = pd.to_datetime(df_plot["date"], errors="coerce")

if df_plot["date"].isna().any():
    bad = df_plot[df_plot["date"].isna()].head()
    raise ValueError(f"Dates invalides dans df_ar12_forecasts['date']:\n{bad}")

df_plot = df_plot.sort_values(["series_id", "date"]).reset_index(drop=True)

# 1) check unicité (important pour plot_series)
dup = df_plot.duplicated(subset=["series_id", "date"]).sum()
if dup > 0:
    # on garde la dernière ligne (cutoff le plus récent) si jamais
    df_plot = (
        df_plot.sort_values(["series_id", "date", "cutoff"])
               .drop_duplicates(subset=["series_id", "date"], keep="last")
               .reset_index(drop=True)
    )

# 2) colonnes PI robustes
lo_candidates = ["y_hat_ar12_lo_95", "y_hat_ar12_lo_95.0"]
hi_candidates = ["y_hat_ar12_hi_95", "y_hat_ar12_hi_95.0"]

lo_col = next((c for c in lo_candidates if c in df_plot.columns), None)
hi_col = next((c for c in hi_candidates if c in df_plot.columns), None)

if lo_col is None or hi_col is None:
    raise KeyError(f"Colonnes PI introuvables. Colonnes dispo: {list(df_plot.columns)}")

# ----------------------------
# Prepare obs
# ----------------------------
df_obs = (
    df_plot.rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_obs": "y",
    })[["unique_id", "ds", "y"]]
)

# ----------------------------
# Prepare forecast + PI (AR12)
# ----------------------------
df_fcst = (
    df_plot.rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_hat_ar12": "AR12",
        lo_col: "AR12-lo-95",
        hi_col: "AR12-hi-95",
    })[["unique_id", "ds", "AR12", "AR12-lo-95", "AR12-hi-95"]]
)

# ----------------------------
# Plot
# ----------------------------
fig = plot_series(
    df=df_obs,
    forecasts_df=df_fcst,
    level=[95],
    engine="plotly",
).update_layout(height=400)

for trace in fig.data:
    if trace.name == "y":
        trace.name = "Unemployment rate (stationary)"
    elif trace.name == "AR12":
        trace.name = "AR(12) univariate"
    elif "level_95" in (trace.name or "").lower():
        trace.name = "95% Prediction Interval"

fig.show()

In [77]:
# ----------------------------
# Prepare forecast + PI (AR12)
# ----------------------------
df_fcst = (
    df_ar12_forecasts
    .rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_hat_ar12": "AR12",
        "y_hat_ar12_lo_95": "AR12-lo-95",
        "y_hat_ar12_hi_95": "AR12-hi-95",
    })
    [[
        "unique_id",
        "ds",
        "AR12",
        "AR12-lo-95",
        "AR12-hi-95",
    ]]
)

# Evaluaer 

In [37]:
import numpy as np
import pandas as pd

# -----------------------------
# 1) One forecast per target date ds
#    (keep the most recent cutoff for each ds)
# -----------------------------
bkt_ar12_final = (
    bkt_ar12_eval
    .sort_values(["unique_id", "ds", "cutoff"])
    .groupby(["unique_id", "ds"], as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

# -----------------------------
# 2) Segments + ALL
# -----------------------------
segments = [
    ("1990-01-01", "1999-12-31", "1990-1999"),
    ("2000-01-01", "2008-07-31", "2000-2008"),
    ("2008-08-01", "2019-12-31", "2008-2019"),
    ("2020-01-01", None,         "2020-fin"),
]

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))

rows = []

# --- ALL periods ---
rows.append({
    "period": "ALL",
    "n_obs": int(len(bkt_ar12_final)),
    "MAE_AR12": mae(bkt_ar12_final["y"], bkt_ar12_final["AR12"]) if len(bkt_ar12_final) > 0 else np.nan,
})

# --- by segments ---
for start, end, label in segments:
    mask = bkt_ar12_final["ds"] >= pd.Timestamp(start)
    if end is not None:
        mask &= bkt_ar12_final["ds"] <= pd.Timestamp(end)

    df_seg = bkt_ar12_final.loc[mask]

    rows.append({
        "period": label,
        "n_obs": int(len(df_seg)),
        "MAE_AR12": mae(df_seg["y"], df_seg["AR12"]) if len(df_seg) > 0 else np.nan,
    })

df_mae_ar12 = pd.DataFrame(rows)

# Optionnel: ordre propre
order = ["ALL"] + [s[2] for s in segments]
df_mae_ar12["period"] = pd.Categorical(df_mae_ar12["period"], categories=order, ordered=True)
df_mae_ar12 = df_mae_ar12.sort_values("period").reset_index(drop=True)

df_mae_ar12

,period,n_obs,MAE_AR12
0,ALL,428,0.678619
1,1990-1999,120,0.408856
2,2000-2008,103,0.395212
3,2008-2019,137,0.521106
4,2020-fin,68,1.901288


Très bien partie

# Sauvegarde

In [32]:
# ============================================================
# (ADD) 6) Save artifacts & outputs (Linear Regression)
# ============================================================

from datetime import datetime
import json

# ----------------------------
# Model identification
# ----------------------------
SERIES_ID = "UNRATE"
MODEL_TAG = "lr_lag12_exog"   # 🔑 clair et extensible

# ----------------------------
# Directories
# ----------------------------
OUTPUT_FORECASTS_DIR = PROJECT_ROOT / "outputs" / "forecasts"

ARTIFACT_CONFIGS_DIR = PROJECT_ROOT / "artifacts" / "configs"
ARTIFACT_CV_DIR      = PROJECT_ROOT / "artifacts" / "cv"
ARTIFACT_META_DIR    = PROJECT_ROOT / "artifacts" / "metadata"

OUTPUT_FORECASTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_CV_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_META_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# 1) Outputs (OOS forecasts)
# ----------------------------
oos_path = OUTPUT_FORECASTS_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_oos_forecasts.parquet"
df_lr_forecasts.to_parquet(oos_path, index=False)

# ----------------------------
# 2) Artifacts – raw backtesting output
# ----------------------------
bkt_path = ARTIFACT_CV_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_bkt_raw.parquet"
bkt_lr.to_parquet(bkt_path, index=False)

# ----------------------------
# 3) Run configuration (reproducibility)
# ----------------------------
run_config = {
    "model": "LinearRegression",
    "framework": "Nixtla-MLForecast",
    "target": SERIES_ID,
    "stationary": True,
    "lags": [12],
    "exogenous_variables": [
        c for c in ts_lr.columns if c not in ["unique_id", "ds", "y"]
    ],
    "horizon": H,
    "step_size": STEP_SIZE,
    "partitions": PARTITIONS,
    "prediction_intervals": {
        "method": "conformal_distribution",
        "levels": LEVELS,
        "n_windows": PI_WINDOWS,
    },
    "frequency": FREQ,
}

cfg_path = ARTIFACT_CONFIGS_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_config.json"
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

# ----------------------------
# 4) Metadata – run info
# ----------------------------
meta_path = ARTIFACT_META_DIR / f"{SERIES_ID.lower()}_{MODEL_TAG}_run_info.json"
run_info = {
    "run_utc": datetime.utcnow().isoformat(),
    "project_root": str(PROJECT_ROOT.resolve()),
    "model_tag": MODEL_TAG,
    "files": {
        "oos_forecasts": str(oos_path.resolve()),
        "cv_raw": str(bkt_path.resolve()),
        "config": str(cfg_path.resolve()),
    },
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2)

print("✅ Saved:")
print(" - OOS forecasts :", oos_path.name)
print(" - CV raw        :", bkt_path.name)
print(" - Config        :", cfg_path.name)
print(" - Metadata      :", meta_path.name)


NameError: name 'df_lr_forecasts' is not defined

# Graphique

## Résultat
On voit notre modèle AR1 est très basique, il suit juste la direction du taux de chômage. Avec ce baseline, on s'aperçoit des informations très importantes pour orienter notre expérimentation. 

De 1990 à 2007, l'économie américaine a été stable. En 2008, elle a été frappée par la crise de Subprime. En 2019, çà été la crise de Coronavirus. Qu'est-ce qu'on peut dire de ces trois périodes? 

Globalement, le modèle auto-régressif reste proche des observations en période de stabilité. C'est une bonne capacité à capter la dynamique du chômage.

Lors des ruptures structurelles des deux crises, la qualité des prévisions se dégrade et les intervalles de prédiction s’élargissent. Ce qui traduit une incertitude de plus en plus élevée.

Le modèle capte la direction des variations, mais sa fiabilité diminue en période de crise, sans masquer cette incertitude.

Cette étude constitue un **sanity check du système de prévision**. Elle valide le comportement attendu du modèle et la cohérence du pipeline.

Une approche plus complexe est attendu. 

## Next
Essayons d'optimiser le paramètre "p" de AR pour confirmer notre analyse. 